## Flow Cytometry Analysis — Combined 48h Experiments Raw Percentage Barplots
### Author: Eleni Aretaki
### Date: 07-09-2026
### Purpose: 
    1. Load two 48h flow cytometry datasets.
    2. Generate phenotype-specific barplots following the heatmaps' order, comparing WT to KO and drug treatment to DMSO-treatment for each one, respectively.
#### Cell Lines: ARID1A, ARID1B, ARID2, SMARCA4, BRD9, WT (plus others partially screened)
#### Compounds: Camptothecin, MMS, Adavosertib, Palbociclib, MLN4924, Hydroxyurea, BIBR1523, Cobimetinib, Lapatinib, Niraparib, Paclitaxe

In [ ]:
import os
import colorsys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.patches import Rectangle
import matplotlib.colors as mc
from matplotlib.gridspec import GridSpec

In [ ]:
# ----------------------------
# Read in raw-percentage dataset 
# (double-positives added on Apoptosis and DSB-positives)
# ----------------------------
fc_complete = pd.read_csv('../48h_combined_raw_data_wo_outliers.csv')

In [ ]:
# ====================================================
# PHENOTYPES
# ====================================================

phenotypes = ['G1-phase', 
              'S-phase', 
              'G2-phase', 
              'Apoptosis-Positive', 
              'DSB-Positive']

# ====================================================
# ORDERS
# ====================================================

cell_order = [
    "C631",
    "ARID1A KO",
    "ARID1B KO",
    "ARID2 KO",
    "PBRM1 KO",
    "BRD9 KO",
    "SMARCC1 KO",
    "SMARCA4 KO"
]

treatment_order = [
    "PBC", "HU", "Ada", "PAC", "LAP",
    "COB", "MMS", "CPT", "MLN", "BIBR"
]


# ====================================================
# COLORS
# ====================================================

color_mapping = {
    "C631": "#9C9B9B",
    "ARID1A KO": "#C52030",
    "ARID1B KO": "#8A181A",
    "ARID2 KO": "#176533",
    "PBRM1 KO": "#93C13E",
    "BRD9 KO": "#D2AE2A",
    "SMARCC1 KO": "#3569A1",
    "SMARCA4 KO": "#80519B"
}


def lighten_color(color, amount=0.45):
    try:
        c = mc.cnames[color]
    except:
        c = color

    r, g, b = mc.to_rgb(c)
    h, l, s = colorsys.rgb_to_hls(r, g, b)

    return colorsys.hls_to_rgb(h, 1 - amount * (1 - l), s)


# ====================================================
# RAW DATA ONLY (NO PRE-AGGREGATION)
# ====================================================

plot_data = fc_complete[
    fc_complete["modification"].isin(cell_order)
].copy()

plot_data = plot_data[
    plot_data["treatment"].isin(["DMSO"] + treatment_order)
]

plot_data["modification"] = pd.Categorical(
    plot_data["modification"],
    categories=cell_order,
    ordered=True
)

plot_data["treatment"] = pd.Categorical(
    plot_data["treatment"],
    categories=["DMSO"] + treatment_order,
    ordered=True
)


# ====================================================
# MATCHING FUNCTION (RETURNS RAW REPLICATES)
# ====================================================

def get_matched_values(phenotype, ko, treatment, plot_data):

    ko_treated = plot_data[
        (plot_data["modification"] == ko) &
        (plot_data["treatment"] == treatment)
    ]

    if ko_treated.empty:
        return None

    output = []

    for _, row in ko_treated.iterrows():

        fc = row["FC"]
        dmso = row["DMSO_conc"]

        wt_dmso = plot_data[
            (plot_data.FC == fc) &
            (plot_data.DMSO_conc == dmso) &
            (plot_data.modification == "C631") &
            (plot_data.treatment == "DMSO")
        ][phenotype].values

        wt_tx = plot_data[
            (plot_data.FC == fc) &
            (plot_data.DMSO_conc == dmso) &
            (plot_data.modification == "C631") &
            (plot_data.treatment == treatment)
        ][phenotype].values

        ko_dmso = plot_data[
            (plot_data.FC == fc) &
            (plot_data.DMSO_conc == dmso) &
            (plot_data.modification == ko) &
            (plot_data.treatment == "DMSO")
        ][phenotype].values

        ko_tx = plot_data[
            (plot_data.FC == fc) &
            (plot_data.DMSO_conc == dmso) &
            (plot_data.modification == ko) &
            (plot_data.treatment == treatment)
        ][phenotype].values

        if len(wt_dmso) == 0 or len(wt_tx) == 0 or len(ko_dmso) == 0 or len(ko_tx) == 0:
            continue

        output.append([wt_dmso, wt_tx, ko_dmso, ko_tx])

    if len(output) == 0:
        return None

    return output


# ====================================================
# PLOTTING
# ====================================================

def safe_mean(arr):
    return np.mean(np.concatenate(arr))


def safe_sem(arr):
    flat = np.concatenate(arr)
    if len(flat) <= 1:
        return 0
    return np.std(flat, ddof=1) / np.sqrt(len(flat))

In [ ]:
# ====================================================
# PLOTTING
# ====================================================

rename_drugs = {
    "PBC": "Palbociclib",
    "HU": "Hydroxyurea",
    "Ada": "Adavosertib",
    "PAC": "Paclitaxel",
    "LAP": "Lapatinib",
    "COB": "Cobimetinib",
    "MMS": "MMS",
    "CPT": "Camptothecin",
    "MLN": "MLN4924",
    "BIBR": "BIBR1532"
}

def plot_a4_phenotype(phenotype, plot_data):

    fig = plt.figure(figsize=(8.27, 11.69))

    gs = GridSpec(
        len(treatment_order),
        len(cell_order) - 1,
        figure=fig,
        left=0.15,
        right=0.98,
        bottom=0.10,
        top=0.95,
        hspace=0.35,
        wspace=0.35
    )

    for row_i, treatment in enumerate(treatment_order):
        for col_i, ko in enumerate(cell_order[1:]):

            ax = fig.add_subplot(gs[row_i, col_i])

            vals = get_matched_values(phenotype, ko, treatment, plot_data)

            # =================================================
            # NOT TESTED
            # =================================================
            if vals is None:
                ax.text(
                    0.5, 0.5,
                    "NOT\nTESTED",
                    ha="center",
                    va="center",
                    fontsize=6,
                    color="gray",
                    fontweight="bold"
                )

                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                continue

            # =================================================
            # FLATTEN ALL REPLICATES
            # =================================================

            means = np.array([
                safe_mean(v) for v in zip(*vals)
            ])

            sems = np.array([
                safe_sem(v) for v in zip(*vals)
            ])

            base = color_mapping[ko]

            colors = [
                lighten_color(color_mapping["C631"]),
                color_mapping["C631"],
                lighten_color(base),
                base
            ]

            ax.bar(
                [0, 1, 2, 3],
                means,
                yerr=sems,
                color=colors,
                capsize=2,
                width=0.6,
                error_kw={"elinewidth": 0.6, "capthick": 0.6}
            )

            if row_i == len(treatment_order) - 1:

                ax.set_xticks([0, 1, 2, 3])

                ax.set_xticklabels(
                    [
                        "WT\nDMSO",
                        "WT\nTreated",
                        "KO\nDMSO",
                        "KO\nTreated"
                    ],
                    fontsize=5,
                    rotation=90
                )

                ax.tick_params(
                    axis="x",
                    length=0,
                    pad=2
                )

            else:

                ax.set_xticks([])
            ax.tick_params(axis="y", labelsize=7, width=0.4, length=2)

            for spine in ax.spines.values():
                spine.set_linewidth(0.4)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            if col_i == 0:
                ax.set_ylabel(rename_drugs.get(treatment, treatment), 
                              fontsize=9, rotation=0,
                              ha="right", va="center")

            if row_i == 0:
                label = ko.replace(" KO", "").replace("C631", "WT")
                ax.set_title(label, fontsize=10, pad=8)

    fig.suptitle(phenotype, fontsize=14, y=0.995)
    fig.supylabel('Fraction of cells (%)', fontsize=10, x=0.0003)

    plt.savefig(
        os.path.join(out_dir_barplots, f"{phenotype}_A4_barplots.pdf"),
        dpi=600,
        bbox_inches="tight"
    )

    plt.close()

In [ ]:
# ====================================================
# RUN
# ====================================================

out_dir_barplots = r"..\Barplots"
os.makedirs(out_dir_barplots, exist_ok=True)

for phenotype in phenotypes:
    print(f"Plotting {phenotype}")
    plot_a4_phenotype(phenotype, plot_data)

print("Done.")

### Barplots of single comparison (one condition only to make the difference between effects on DSB, PARP and Double positives more distinct)

In [ ]:
# ----------------------------
# Read in raw-percentage datasets 
# (double-positives as separate phenotype)
# ----------------------------
fc_1 = pd.read_csv("../fc1_combined_data_wo_dp.csv")
fc_2 = pd.read_csv("../fc2_combined_data_wo_dp.csv")

fc_1["FC"] = 1
fc_2["FC"] = 2

# Remove treatments that were expired or inactive
fc_1 = fc_1[fc_1["treatment"] != "Aph"]
fc_1["treatment"] = fc_1["treatment"].str.replace("Pbc", "PBC")
fc_2 = fc_2[fc_2["treatment"] != "NIR"]

# Combine the two experiments
fc_complete_dp = pd.concat([fc_1, fc_2], ignore_index=True)

In [ ]:
# ====================================================
# PHENOTYPES
# ====================================================

phenotypes_dp = ['G1-phase', 
              'S-phase', 
              'G2-phase', 
              'Apoptosis-Positive', 
              'DSB-Positive',
             'Double-Positive']

# ====================================================
# RAW DATA ONLY (NO PRE-AGGREGATION)
# ====================================================

plot_data_dp = fc_complete_dp[
    fc_complete_dp["modification"].isin(cell_order)
].copy()

plot_data_dp = plot_data_dp[
    plot_data_dp["treatment"].isin(["DMSO"] + treatment_order)
]

plot_data_dp["modification"] = pd.Categorical(
    plot_data_dp["modification"],
    categories=cell_order,
    ordered=True
)

plot_data_dp["treatment"] = pd.Categorical(
    plot_data_dp["treatment"],
    categories=["DMSO"] + treatment_order,
    ordered=True
)

In [ ]:
def plot_single_comparison(phenotype, ko, treatment, plot_data, out_dir_barplots):

    os.makedirs(out_dir_barplots, exist_ok=True)
    fig, ax = plt.subplots(figsize=(3.5,4.5))

    vals = get_matched_values(
        phenotype,
        ko,
        treatment,
        plot_data
    )

    if vals is None:
        print(f"No data for {ko} {treatment}")
        return

    means = np.array([
        safe_mean(v) for v in zip(*vals)
    ])

    sems = np.array([
        safe_sem(v) for v in zip(*vals)
    ])

    colors = [
        lighten_color(color_mapping["C631"]),
        color_mapping["C631"],
        lighten_color(color_mapping[ko]),
        color_mapping[ko]
    ]

    ax.bar(
        [0,1,2,3],
        means,
        yerr=sems,
        color=colors,
        capsize=2,
        width=0.6,
        error_kw={"elinewidth": 0.6, "capthick": 0.6}
    )

    ax.set_xticks([0,1,2,3])
    ax.set_xticklabels(
        [
            "WT\nDMSO",
            "WT\nCPT",
            "SMARCA4\nDMSO",
            "SMARCA4\nCPT"
        ],
        fontsize=8
    )

    ax.set_ylabel("Fraction of cells (%)")

    ax.set_title(phenotype)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            out_dir_barplots,
            f"{ko}_{treatment}_{phenotype}.pdf"
        ),
        dpi=600,
        bbox_inches="tight"
    )

    plt.close()

In [ ]:
for phenotype in phenotypes_dp:
    plot_single_comparison(
        phenotype=phenotype,
        ko="SMARCA4 KO",
        treatment="CPT",
        plot_data=plot_data_dp,
        out_dir_barplots = r"..\Barplots\DP_separate_SMARCA4_CPT_barplots"
    )